# 06 — Export des pipelines GroupKFold 16/médiane

Complément demandé : conserver les deux protocoles avec leurs modèles complets.
Même configuration que 05 : 16 entrées numériques + procédé, médiane, Charpy
avec température connue. Les anciens scores 13/itérative de 03 restent historiques.

Méthode : vérifier les hashes de 02/04, réajuster les quatre familles avec leurs
paramètres **déjà sélectionnés dans les plis internes groupés de 04** et reproduire
leurs 80 évaluations OOF (toutes leurs prédictions). Calculer aussi la baseline médiane avec ces plis.
Choisir par MAE interne, exporter 20 modèles de trains externes, puis régler les
cinq familles avec GroupKFold5 sur tous les labels et exporter 20 refits.
Chaque transformation est ajustée au train ; aucun score externe ne sert au choix.
Les refits complets servent à l'inférence, jamais à une future évaluation de plis.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from joblib import Memory, parallel_config
from threadpoolctl import threadpool_limits

SEED = 42
N_JOBS = 3  # recherches parallèles ; chaque forêt utilise un seul worker
PRIMARY_SET = 'A1b_16'
PRIMARY_IMPUTER = 'median'
TARGETS = ['yield_strength', 'uts', 'elongation', 'charpy_J']
CAT = 'weld_process'
PROCESS_COL = CAT
CORE13 = ['C', 'Si', 'Mn', 'S', 'P', 'current_A', 'voltage_V',
          'heat_input_kJmm', 'interpass_C', 'pwht_temp_C', 'pwht_time_h',
          'O_ppm', 'N_ppm']
CORE16 = CORE13 + ['Ni', 'Cr', 'Mo']
NONNEG = CORE16  # la température d'essai peut être négative
TARGET_UNITS = {'yield_strength': 'MPa', 'uts': 'MPa',
                'elongation': 'points de %', 'charpy_J': 'J'}
TARGET_NAMES = {'yield_strength': "Limite d'élasticité", 'uts': 'Résistance maximale',
                'elongation': 'Allongement', 'charpy_J': 'Énergie Charpy — exploratoire'}
MODEL_NAMES = {'median_y': 'Médiane de y', 'ridge': 'Ridge', 'svr_rbf': 'SVR RBF',
               'random_forest': 'Random Forest', 'knn': 'kNN'}
PARAM_GRIDS = {
    'median_y': {},
    'ridge': {'model__alpha': [0.01, 0.1, 1, 10, 100, 1000]},
    'svr_rbf': {'model__regressor__C': [1, 10, 100],
                'model__regressor__gamma': ['scale', 0.05],
                'model__regressor__epsilon': [0.05, 0.2]},
    'random_forest': {'model__min_samples_leaf': [1, 2, 5],
                     'model__max_depth': [None, 10],
                     'model__max_features': [0.5, 1.0]},
    'knn': {'model__n_neighbors': [3, 5, 9, 15], 'model__p': [1, 2]},
}

def numeric_features(target):
    if target not in TARGETS:
        raise ValueError(f'Cible non prévue : {target}')
    return list(CORE16) + (['charpy_temp_C'] if target == 'charpy_J' else [])

def task_features(target):
    return numeric_features(target) + [CAT]

def eligible_rows(frame, target):
    # La température d'essai est une condition connue pour E(T), jamais imputée
    # pour fabriquer un couple Charpy dont la condition d'essai serait inconnue.
    mask = frame['in_strict13'] & frame[target].notna()
    if target == 'charpy_J':
        mask &= frame.charpy_temp_C.notna()
    return frame.loc[mask].copy()

def regression_metrics(y, prediction):
    y, prediction = np.asarray(y, float), np.asarray(prediction, float)
    if not np.isfinite(y).all() or not np.isfinite(prediction).all():
        raise ValueError('Labels/prédictions non finis.')
    return {'MAE': float(mean_absolute_error(y, prediction)),
            'RMSE': float(mean_squared_error(y, prediction) ** 0.5),
            'R2': float(r2_score(y, prediction)) if np.ptp(y)>0 else np.nan}

from sklearn.model_selection import KFold, GroupKFold
from scipy.stats import spearmanr
import cloudpickle
import copy
import subprocess
import sys
import os

def condition_signature(frame,columns):
    # Exact available values, including an explicit unknown marker; never impute.
    # Equality of partial signatures is a proxy, not proof of the same specimen.
    def token(v):
        if pd.isna(v): return None
        return v.item() if isinstance(v,np.generic) else v
    return frame[columns].apply(lambda row:json.dumps([token(v) for v in row],
                               ensure_ascii=False,separators=(',',':')),axis=1)

def train_seen_signatures(train,valid,columns):
    return condition_signature(valid,columns).isin(set(condition_signature(train,columns)))

def detached_pipeline(fitted):
    model=copy.deepcopy(fitted)
    model.memory=None
    return model

def dump_pipeline(fitted,path,metadata,probe):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    exported=detached_pipeline(fitted)
    with path.open('wb') as stream: cloudpickle.dump(exported,stream,protocol=5)
    with path.open('rb') as stream: loaded=cloudpickle.load(stream)
    np.testing.assert_allclose(loaded.predict(probe),fitted.predict(probe),rtol=1e-10,atol=1e-8)
    metadata=dict(metadata,artifact_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                  serialization='cloudpickle_protocol5',versions=VERSIONS)
    path.with_suffix('.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False)+'\n')
    return metadata
def outer_partitions(frame, target, outer_fold):
    eligible = eligible_rows(frame, target)
    train = eligible.loc[eligible.outer_fold != outer_fold].reset_index(drop=True)
    valid = eligible.loc[eligible.outer_fold == outer_fold].reset_index(drop=True)
    if train.empty or valid.empty:
        raise ValueError(f'Partition vide : {target}, pli {outer_fold}')
    if not set(train.group).isdisjoint(valid.group):
        raise AssertionError('Un groupe externe est partagé entre train et validation.')
    return train, valid

def inner_splits(train, outer_fold):
    col = f'inner_fold_when_outer{outer_fold}'
    if not set(train[col].unique()).issubset({0, 1, 2}):
        raise AssertionError('Les lignes externes ne doivent pas entrer dans les plis internes.')
    splits = []
    for inner_fold in range(3):
        tr = np.flatnonzero(train[col].to_numpy() != inner_fold)
        va = np.flatnonzero(train[col].to_numpy() == inner_fold)
        if len(tr) < 15 or len(va) < 2:
            raise ValueError('Effectif interne insuffisant pour les grilles prévues.')
        if not set(train.iloc[tr].group).isdisjoint(train.iloc[va].group):
            raise AssertionError('Un groupe interne est partagé.')
        splits.append((tr, va))
    return splits


def grouped_full_splits(train):
    partitions=list(GroupKFold(5).split(train,groups=train.group))
    for tr,va in partitions:
        assert set(train.iloc[tr].group).isdisjoint(train.iloc[va].group)
    return partitions

def grouped_access(frame,target,outer,inner=None):
    candidate=frame.loc[frame.in_strict13 & ~frame.is_duplicate_excess].copy()
    if target=='charpy_J': candidate=candidate.loc[candidate.charpy_temp_C.notna()]
    allowed=candidate.outer_fold!=outer
    valid=candidate.outer_fold==outer
    if inner is not None:
        allowed &= candidate[f'inner_fold_when_outer{outer}']!=inner
        valid=(candidate.outer_fold!=outer)&(candidate[f'inner_fold_when_outer{outer}']==inner)
    labeled=candidate[target].notna()
    training=candidate.loc[allowed & labeled]
    pool=candidate.loc[allowed & ~labeled]
    validation=candidate.loc[valid & labeled]
    assert set(training.group).isdisjoint(validation.group)
    assert set(pool.group).isdisjoint(validation.group)
    assert set(training.raw_row_id).isdisjoint(pool.raw_row_id)
    return training,pool,validation


## Identité, colonnes et sources des réglages

N/encodages/quarantaines viennent du master préparé. C/Si/Mn/S/P/Ni/Cr/Mo :
% massique ; O/N : ppm ; courant A, tension V, apport thermique kJ/mm,
températures entre passes/PWHT °C, durée PWHT h. Procédé catégoriel ;
Charpy ajoute sa température °C et prédit J. Traction : MPa/MPa/%.
Pas d'ID/groupe, autre cible, dureté/FATT50 ou microstructure dans X.


In [2]:
PROJECT_ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'sujet.md').is_file())
DATA=PROJECT_ROOT/'data/processed/modeling'
DIAGNOSTICS=PROJECT_ROOT/'data/processed/experiments/salh_diagnostics_v1'
OUT=PROJECT_ROOT/'data/processed/experiments/salh_group_kfold_v1'
MODELS=PROJECT_ROOT/'models/salh_group_kfold_v1'
OUT.mkdir(parents=True,exist_ok=True); MODELS.mkdir(parents=True,exist_ok=True)
STARTED=time.perf_counter()
VERSIONS={'python':platform.python_version(),'sklearn':sklearn.__version__,
          'numpy':np.__version__,'pandas':pd.__version__,'joblib':joblib.__version__,
          'cloudpickle':cloudpickle.__version__}
preparation_manifest=json.loads((DATA/'manifest.json').read_text())
diagnostic_manifest=json.loads((DIAGNOSTICS/'manifest.json').read_text())
for folder,manifest in [(DATA,preparation_manifest),(DIAGNOSTICS,diagnostic_manifest)]:
    for filename,digest in manifest['files'].items():
        assert hashlib.sha256((folder/filename).read_bytes()).hexdigest()==digest,filename
assert hashlib.sha256((PROJECT_ROOT/'data/raw/welddb.data').read_bytes()).hexdigest()==preparation_manifest['raw_sha256']
master=pd.read_csv(DATA/'master.csv')
group_folds=pd.read_csv(DATA/'folds.csv')
data=master.loc[master.in_strict13 & ~master.is_duplicate_excess].merge(group_folds,on=['raw_row_id','group'],validate='one_to_one')
assert len(data)==1555 and data.group.nunique()==44
def save_csv(frame,name):
    frame.to_csv(OUT/name,index=False); return frame
def show_saved(frame,name,limit=25):
    save_csv(frame,name); display(frame.head(limit))
counts=pd.DataFrame([{'target':t,'labels':len(eligible_rows(data,t)),
    'groups_labeled':eligible_rows(data,t).group.nunique(),'unit':TARGET_UNITS[t],
    'unlabeled_X_available':int((data[t].isna() & (data.charpy_temp_C.notna() if t=='charpy_J' else True)).sum())}
    for t in TARGETS])
show_saved(counts,'target_counts.csv')
show_saved(pd.DataFrame({'feature':CORE16,'missing_fraction':data[CORE16].isna().mean().to_numpy()}),'input_missingness.csv')
reference_metrics=pd.read_csv(DIAGNOSTICS/'sensitivity_outer_metrics.csv')
reference_predictions=pd.read_csv(DIAGNOSTICS/'sensitivity_outer_predictions.csv')
reference_inner=pd.read_csv(DIAGNOSTICS/'sensitivity_inner_search.csv')
def current_config(frame):
    return frame.loc[(frame.feature_set==PRIMARY_SET)&(frame.imputer==PRIMARY_IMPUTER)].copy()
reference_metrics=current_config(reference_metrics)
reference_predictions=current_config(reference_predictions)
reference_inner=current_config(reference_inner)
assert len(reference_metrics)==80 and len(reference_inner)==760


,target,labels,groups_labeled,unit,unlabeled_X_available
0,yield_strength,706,40,MPa,849
1,uts,677,39,MPa,878
2,elongation,631,34,points de %,924
3,charpy_J,861,24,J,0


,feature,missing_fraction
0,C,0.000000
1,Si,0.000000
2,Mn,0.000000
3,S,0.000000
4,P,0.000000
5,current_A,0.114469
6,voltage_V,0.114469
7,heat_input_kJmm,0.000000
8,interpass_C,0.000000
9,pwht_temp_C,0.000000


## Pipelines autonomes

Statistiques numériques et catégories apprises dans chaque train → one-hot du
procédé → médiane train → garde PWHT sur les couples compatibles observés.
SVR standardise y dans son train ; RF300 seed42 ; kNN uniforme L1/L2 ; Ridge L2.
Baseline : seule médiane de y. Mêmes grilles et critères que 05.


In [3]:
class FoldPreprocessor(BaseEstimator, TransformerMixin):
    """À ajuster sur les lignes d'entraînement d'un pli uniquement.

    1. standardisation des entrées numériques (moyenne/écart-type du train, NaN ignorés) ;
    2. one-hot du procédé (catégories du train ; inconnue -> vecteur nul) ;
    3. imputation : 'iterative' (IterativeImputer/BayesianRidge, init médiane, ordre ascendant,
       max_iter=100, tol=1e-3, random_state=42, bornes physiques >= 0) ou 'median' ;
    4. PWHT : un couple (température, durée) imputé est projeté sur le couple réel du train
       le plus proche (distance en écarts-types du train), en respectant une valeur observée.
    Une colonne entièrement absente du train est refusée plutôt que fabriquée.
    Sortie : entrées standardisées + indicatrices du procédé. Les cibles n'entrent jamais ici.
    """

    def __init__(self, num_cols, cat_col=PROCESS_COL, method='median', max_iter=100,
                 tol=1e-3, random_state=SEED):
        self.num_cols, self.cat_col, self.method = num_cols, cat_col, method
        self.max_iter, self.tol, self.random_state = max_iter, tol, random_state

    def _encode(self, X):
        if 'charpy_temp_C' in self.num_cols and X['charpy_temp_C'].isna().any():
            raise ValueError("Température Charpy requise ; aucune condition d'essai ne peut être imputée.")
        Z = (X[self.num_cols].to_numpy(float) - self.mu_) / self.sd_
        C = np.stack([(X[self.cat_col] == c).to_numpy(float) for c in self.cats_], axis=1) \
            if self.cat_col else np.empty((len(X), 0))
        return np.hstack([Z, C])

    def fit(self, X, y=None):
        Xn = X[self.num_cols].astype(float)
        empty = Xn.columns[Xn.notna().sum() == 0].tolist()
        if empty:
            raise ValueError(f'Colonnes entièrement absentes du train : {empty}')
        self.mu_ = Xn.mean().to_numpy()
        self.sd_ = Xn.std(ddof=0).replace(0, 1).to_numpy()
        self.cats_ = sorted(X[self.cat_col].dropna().unique()) if self.cat_col else []
        Z = self._encode(X)
        lo = np.full(Z.shape[1], -np.inf)
        for i, c in enumerate(self.num_cols):
            if c in NONNEG:
                lo[i] = (0 - self.mu_[i]) / self.sd_[i]
        if self.method == 'iterative':
            self.imp_ = IterativeImputer(estimator=BayesianRidge(), initial_strategy='median',
                                         imputation_order='ascending', max_iter=self.max_iter,
                                         tol=self.tol, random_state=self.random_state,
                                         skip_complete=False, min_value=lo, keep_empty_features=True)
        elif self.method == 'median':
            self.imp_ = SimpleImputer(strategy='median', keep_empty_features=True)
        else:
            raise ValueError(self.method)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', ConvergenceWarning)
            self.imp_.fit(Z)
        self.converged_ = (self.method != 'iterative') or (self.imp_.n_iter_ < self.max_iter)
        self.n_iter_ = getattr(self.imp_, 'n_iter_', 0)
        if not self.converged_:
            raise RuntimeError('Imputation itérative non convergée : réviser le schéma dans le train.')
        # couples PWHT réels du train
        self.pw_idx_ = [self.num_cols.index(c) for c in ('pwht_temp_C', 'pwht_time_h')
                        if c in self.num_cols]
        if len(self.pw_idx_) == 2:
            P = Z[:, self.pw_idx_]
            self.pw_pairs_ = np.unique(P[~np.isnan(P).any(axis=1)], axis=0)
            if len(self.pw_pairs_) == 0:
                raise ValueError('Aucun couple PWHT complet dans le train.')
        return self

    def transform(self, X):
        Z = self._encode(X)
        miss = np.isnan(Z)
        out = self.imp_.transform(Z)
        if len(self.pw_idx_) == 2 and len(self.pw_pairs_):
            i, j = self.pw_idx_
            for r in np.where(miss[:, i] | miss[:, j])[0]:
                cand = self.pw_pairs_
                if not miss[r, i]:   # température observée : couples compatibles seulement
                    cand = cand[np.isclose(cand[:, 0], Z[r, i])]
                if not miss[r, j]:
                    cand = cand[np.isclose(cand[:, 1], Z[r, j])]
                if len(cand) == 0:
                    raise ValueError(f'Aucun couple PWHT compatible pour la ligne {X.index[r]}.')
                best = cand[np.argmin(((cand - out[r, [i, j]]) ** 2).sum(axis=1))]
                out[r, i] = Z[r, i] if not miss[r, i] else best[0]
                out[r, j] = Z[r, j] if not miss[r, j] else best[1]
        return out

    def get_feature_names_out(self, input_features=None):
        return np.array([str(c) for c in self.num_cols] + [f'{self.cat_col}={c}' for c in self.cats_], dtype=object)

In [4]:
def make_pipeline(model_name, target, memory=None):
    if model_name == 'median_y':
        # DummyRegressor ignore les valeurs de X ; seule la médiane de y est apprise.
        return Pipeline([('model', DummyRegressor(strategy='median'))])
    estimators = {
        'ridge': Ridge(),
        'svr_rbf': TransformedTargetRegressor(
            regressor=SVR(kernel='rbf'), transformer=StandardScaler()),
        'random_forest': RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
        'knn': KNeighborsRegressor(weights='uniform'),
    }
    if model_name not in estimators:
        raise ValueError(model_name)
    return Pipeline([
        ('preprocess', FoldPreprocessor(numeric_features(target), cat_col=CAT,
                                       method=PRIMARY_IMPUTER, random_state=SEED)),
        ('model', estimators[model_name]),
    ], memory=memory)


def search_pipeline(train,target,model,cv,memory=None):
    search=GridSearchCV(make_pipeline(model,target,memory),PARAM_GRIDS[model],
        scoring='neg_mean_absolute_error',cv=cv,refit=True,n_jobs=N_JOBS,
        error_score='raise',return_train_score=False)
    with parallel_config(backend='threading'),threadpool_limits(limits=1):
        search.fit(train[task_features(target)],train[target])
    return search

## Reconstitution et export des trains externes groupés

Les paramètres enregistrés sont vérifiés contre le minimum de MAE interne.
La reproduction exacte des prédictions externes vérifie que les nouveaux exports
représentent les modèles déjà évalués. L'ordre des candidats départage les égalités.
La sélection 16/médiane + baseline est distincte de la sélection de schéma/imputer
du diagnostic 04 : aucune nouvelle performance indépendante n'est revendiquée.


In [5]:

replayed_metrics=[]; replayed_predictions=[]; selection_rows=[]; selected_predictions=[]; artifact_rows=[]
for target in TARGETS:
    features=task_features(target)
    for outer in range(5):
        train,valid=outer_partitions(data,target,outer); cv=inner_splits(train,outer)
        models={}; fold_metrics=[]; fold_predictions={}
        for family in PARAM_GRIDS:
            if family=='median_y':
                search=search_pipeline(train,target,family,cv)
                fitted=search.best_estimator_; params=search.best_params_; inner_mae=-search.best_score_
            else:
                match=reference_metrics.loc[(reference_metrics.target==target)&
                    (reference_metrics.outer_fold==outer)&(reference_metrics.model==family)]
                assert len(match)==1
                saved=match.iloc[0]; params=json.loads(saved.best_params); inner_mae=float(saved.inner_MAE)
                inner=reference_inner.loc[(reference_inner.target==target)&
                    (reference_inner.outer_fold==outer)&(reference_inner.model==family)].sort_values('candidate')
                winner=inner.loc[inner.inner_MAE.idxmin()]
                assert params==json.loads(winner.params)
                np.testing.assert_allclose(inner_mae,winner.inner_MAE)
                fitted=make_pipeline(family,target).set_params(**params)
                with threadpool_limits(limits=1): fitted.fit(train[features],train[target])
            pred=fitted.predict(valid[features])
            actual=regression_metrics(valid[target],pred)
            if family!='median_y':
                ref=reference_predictions.loc[(reference_predictions.target==target)&
                    (reference_predictions.outer_fold==outer)&(reference_predictions.model==family)]
                assert ref.raw_row_id.is_unique and set(ref.raw_row_id)==set(valid.raw_row_id)
                aligned=ref.set_index('raw_row_id').loc[valid.raw_row_id]
                np.testing.assert_allclose(aligned.y_true,valid[target],rtol=1e-10,atol=1e-8)
                np.testing.assert_allclose(aligned.y_pred,pred,rtol=1e-10,atol=1e-8)
                for metric in ['MAE','RMSE','R2']: np.testing.assert_allclose(actual[metric],saved[metric])
            row={'target':target,'model':family,'outer_fold':outer,'inner_MAE':inner_mae,
                 'best_params':json.dumps(params,sort_keys=True),'n_train':len(train),'n_valid':len(valid),
                 'verified_previous_OOF':family!='median_y',**actual}
            pp=valid[['raw_row_id','group','outer_fold']].assign(target=target,model=family,
                y_true=valid[target].to_numpy(),y_pred=pred)
            replayed_metrics.append(row); replayed_predictions.append(pp); fold_metrics.append(row)
            models[family]=fitted; fold_predictions[family]=pp
        chosen=min(fold_metrics,key=lambda r:r['inner_MAE']); family=chosen['model']
        selected=fold_predictions[family].assign(model='inner_selected')
        selection_rows.append(dict(chosen,model='inner_selected',selected_family=family))
        selected_predictions.append(selected)
        artifact=MODELS/'outer_selected'/f'{target}_fold{outer}.pkl'
        metadata={'role':'outer_train_only','target':target,'family':family,'outer_fold':outer,
            'protocol':'grouped','features':features,'unit':TARGET_UNITS[target],
            'feature_set':PRIMARY_SET,'imputer':PRIMARY_IMPUTER,
            'train_raw_row_ids':train.raw_row_id.tolist(),'valid_raw_row_ids':valid.raw_row_id.tolist(),
            'train_groups':sorted(train.group.unique()),'valid_groups':sorted(valid.group.unique()),
            'best_params':json.loads(chosen['best_params']),'inner_MAE':chosen['inner_MAE'],
            'raw_sha256':preparation_manifest['raw_sha256'],'evaluation':regression_metrics(selected.y_true,selected.y_pred)}
        saved_export=dump_pipeline(models[family],artifact,metadata,valid[features].head(10))
        artifact_rows.append({'role':'outer_train_only','target':target,'family':family,'outer_fold':outer,
            'path':str(artifact.relative_to(PROJECT_ROOT)),'sha256':saved_export['artifact_sha256']})
        save_csv(pd.DataFrame(replayed_metrics),'outer_metrics.csv')
        save_csv(pd.concat(replayed_predictions,ignore_index=True),'outer_predictions.csv')
        save_csv(pd.DataFrame(selection_rows),'inner_selected_metrics.csv')
        save_csv(pd.concat(selected_predictions,ignore_index=True),'inner_selected_predictions.csv')
        print('Export groupé du train externe :',target,outer,family,flush=True)
scores=pd.DataFrame(replayed_metrics); selected=pd.DataFrame(selection_rows)
assert len(scores)==100 and len(selected)==20 and scores.verified_previous_OOF.sum()==80
summary=selected.groupby('target').agg(MAE_mean=('MAE','mean'),MAE_sd=('MAE','std'),
    R2_mean=('R2','mean'),R2_sd=('R2','std')).reset_index()
show_saved(summary,'selected_metrics_summary.csv')
for target,g in pd.concat(selected_predictions).groupby('target'):
    assert g.raw_row_id.is_unique and set(g.raw_row_id)==set(eligible_rows(data,target).raw_row_id)


Export groupé du train externe : yield_strength 0 random_forest


Export groupé du train externe : yield_strength 1 random_forest


Export groupé du train externe : yield_strength 2 svr_rbf


Export groupé du train externe : yield_strength 3 random_forest


Export groupé du train externe : yield_strength 4 svr_rbf


...

Export groupé du train externe : charpy_J 1 svr_rbf


Export groupé du train externe : charpy_J 2 median_y


Export groupé du train externe : charpy_J 3 random_forest


Export groupé du train externe : charpy_J 4 median_y


,target,MAE_mean,MAE_sd,R2_mean,R2_sd
0,charpy_J,35.751798,11.863081,0.074507,0.137931
1,elongation,3.517381,0.515859,0.156516,0.220106
2,uts,54.344495,13.181389,0.333607,0.228326
3,yield_strength,56.756902,12.397757,0.336606,0.160855


## Réglage groupé et refit sur tous les labels

GroupKFold5 sur les séries des labels, sans shuffle ; les groupes de chaque pli
sont vérifiés disjoints. La MAE de sélection n'est pas un nouveau test externe.
Un index retient la meilleure MAE CV par cible. Les hyperparamètres peuvent
différer de ceux choisis par KFold aléatoire, même sur les mêmes lignes finales.


In [6]:
cache=Memory(OUT/'pipeline_cache',verbose=0)
full_search_rows=[]; full_summary=[]; chosen_final=[]
for target in TARGETS:
    train=eligible_rows(data,target).sort_values('raw_row_id').reset_index(drop=True)
    cv=grouped_full_splits(train); features=task_features(target)
    options=[]
    for family in PARAM_GRIDS:
        search=search_pipeline(train,target,family,cv,cache)
        params=search.best_params_
        for i,config in enumerate(search.cv_results_['params']):
            full_search_rows.append({'target':target,'family':family,'candidate':i,
                'params':json.dumps(config,sort_keys=True),'CV_MAE':-search.cv_results_['mean_test_score'][i]})
        artifact=MODELS/'full_fit'/f'{target}_{family}.pkl'
        metadata={'role':'full_labeled_refit','target':target,'family':family,'features':features,
            'unit':TARGET_UNITS[target],'feature_set':PRIMARY_SET,'imputer':PRIMARY_IMPUTER,
            'train_raw_row_ids':train.raw_row_id.tolist(),'best_params':params,'selection_CV_MAE':-search.best_score_,
            'selection_cv':'GroupKFold5 on labeled source groups, shuffle=False','protocol':'grouped','independent_test_evaluated':False,
            'raw_sha256':preparation_manifest['raw_sha256'],'do_not_use_for_fold_evaluation':True}
        saved=dump_pipeline(search.best_estimator_,artifact,metadata,train[features].head(10))
        entry={'role':'full_labeled_refit','target':target,'family':family,'outer_fold':-1,
            'path':str(artifact.relative_to(PROJECT_ROOT)),'sha256':saved['artifact_sha256'],
            'selection_CV_MAE':-search.best_score_,'best_params':json.dumps(params,sort_keys=True)}
        artifact_rows.append(entry); full_summary.append(entry); options.append(entry)
        save_csv(pd.DataFrame(full_search_rows),'full_refit_inner_search.csv')
        print('Pipeline finale exportée :',target,family,'MAE CV=',round(-search.best_score_,3),flush=True)
    chosen_final.append(min(options,key=lambda x:x['selection_CV_MAE']))
show_saved(pd.DataFrame(full_summary),'full_refit_summary.csv',20)
show_saved(pd.DataFrame(chosen_final),'selected_final_models.csv')
save_csv(pd.DataFrame(artifact_rows),'model_artifacts.csv')
(MODELS/'selected_models.json').write_text(json.dumps(chosen_final,indent=2,ensure_ascii=False)+'\n')

Pipeline finale exportée : yield_strength median_y MAE CV= 70.732


Pipeline finale exportée : yield_strength ridge MAE CV= 64.387


Pipeline finale exportée : yield_strength svr_rbf MAE CV= 60.87


Pipeline finale exportée : yield_strength random_forest MAE CV= 58.695


Pipeline finale exportée : yield_strength knn MAE CV= 71.515


...

Pipeline finale exportée : charpy_J random_forest MAE CV= 32.375


Pipeline finale exportée : charpy_J knn MAE CV= 35.639


,role,target,family,outer_fold,path,sha256,selection_CV_MAE,best_params
0,full_labeled_refit,yield_strength,median_y,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,b885314b6ebb3c9d3c194e73c2d61bd17e04aefc6a2999...,70.732123,{}
1,full_labeled_refit,yield_strength,ridge,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,e798aa76d68e67b2e21c581b5af85fa24efa69b009d297...,64.386792,"{""model__alpha"": 100}"
2,full_labeled_refit,yield_strength,svr_rbf,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,54b523bd654c7efdbb601ebebd447695c9f3eef819529d...,60.869661,"{""model__regressor__C"": 10, ""model__regressor_..."
3,full_labeled_refit,yield_strength,random_forest,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,b4205d9c7e0d3347c9f28d8a8108de0b1619f66161cada...,58.694797,"{""model__max_depth"": null, ""model__max_feature..."
4,full_labeled_refit,yield_strength,knn,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,b92a7fb47a82bc193911d452cd5d25fe59db835553653a...,71.515247,"{""model__n_neighbors"": 15, ""model__p"": 1}"
5,full_labeled_refit,uts,median_y,-1,models/salh_group_kfold_v1/full_fit/uts_median...,97ae89d437e4f2123641ef214aff9eebac3fcf61f81ef0...,69.045240,{}
6,full_labeled_refit,uts,ridge,-1,models/salh_group_kfold_v1/full_fit/uts_ridge.pkl,d417fb2ac92e01edd46d338c07ee6111e5e58deee2dc54...,54.209982,"{""model__alpha"": 100}"
7,full_labeled_refit,uts,svr_rbf,-1,models/salh_group_kfold_v1/full_fit/uts_svr_rb...,c3d3a204c0ebecffe17a59f651f3d043e2afcb011fc7e3...,46.555155,"{""model__regressor__C"": 10, ""model__regressor_..."
8,full_labeled_refit,uts,random_forest,-1,models/salh_group_kfold_v1/full_fit/uts_random...,517d787b2bd69e20f0c09699f01026a11d11ac160c7492...,49.216197,"{""model__max_depth"": 10, ""model__max_features""..."
9,full_labeled_refit,uts,knn,-1,models/salh_group_kfold_v1/full_fit/uts_knn.pkl,9bea29f64dae63ad5e4aea33537f942d2286c22740a5e5...,59.581118,"{""model__n_neighbors"": 3, ""model__p"": 1}"


,role,target,family,outer_fold,path,sha256,selection_CV_MAE,best_params
0,full_labeled_refit,yield_strength,random_forest,-1,models/salh_group_kfold_v1/full_fit/yield_stre...,b4205d9c7e0d3347c9f28d8a8108de0b1619f66161cada...,58.694797,"{""model__max_depth"": null, ""model__max_feature..."
1,full_labeled_refit,uts,svr_rbf,-1,models/salh_group_kfold_v1/full_fit/uts_svr_rb...,c3d3a204c0ebecffe17a59f651f3d043e2afcb011fc7e3...,46.555155,"{""model__regressor__C"": 10, ""model__regressor_..."
2,full_labeled_refit,elongation,random_forest,-1,models/salh_group_kfold_v1/full_fit/elongation...,a4969d1e7cd18ad87a80b3bbe9ca10cf7a747ba7df82f0...,3.281767,"{""model__max_depth"": null, ""model__max_feature..."
3,full_labeled_refit,charpy_J,random_forest,-1,models/salh_group_kfold_v1/full_fit/charpy_J_r...,cc2348cac7b7ea1db3bcc86f1e62ad65b15eca090c46c3...,32.374768,"{""model__max_depth"": 10, ""model__max_features""..."


1755

## Rechargement autonome et préparation semi-supervisée

40 rechargements indépendants du notebook ; contrôle des prédictions et groupes.
Pour le semi-supervisé inductif par groupes, X sans y n'est accessible que dans
les groupes du train courant, externe puis interne. Aucun COREG exécuté ici.
Les artefacts sont ignorés par Git, prétraitement inclus, cache désactivé.


In [7]:
reload_rows=[]
loader="""import cloudpickle,pandas as pd,json,sys
from pathlib import Path
p=Path(sys.argv[1]); meta=json.loads(p.with_suffix('.json').read_text())
with p.open('rb') as f: pipeline=cloudpickle.load(f)
x=pd.read_csv(sys.argv[2])[meta['features']]
print(json.dumps(pipeline.predict(x).tolist()))
"""
for entry in chosen_final:
    target=entry['target']; family=entry['family']; path=PROJECT_ROOT/entry['path']
    probe=eligible_rows(data,target).sort_values('raw_row_id').head(10)[task_features(target)].copy()
    probe_path=OUT/f'reload_probe_{target}.csv'; probe.to_csv(probe_path,index=False)
    with path.open('rb') as stream: loaded=cloudpickle.load(stream)
    expected=loaded.predict(pd.read_csv(probe_path))
    env=dict(os.environ,OPENBLAS_NUM_THREADS='1',MPLCONFIGDIR='/tmp/salh-matplotlib')
    result=subprocess.run([sys.executable,'-c',loader,str(path),str(probe_path)],
        check=True,capture_output=True,text=True,env=env)
    actual=np.asarray(json.loads(result.stdout))
    np.testing.assert_allclose(actual,expected,rtol=1e-10,atol=1e-8)
    reload_rows.append({'target':target,'family':family,'fresh_process_verified':True,
                        'max_prediction_difference':float(np.max(np.abs(actual-expected)))})
show_saved(pd.DataFrame(reload_rows),'fresh_reload_checks.csv')

# Independently loaded models reproduce all held-out predictions, not just probes.
verifier="""import json,cloudpickle,pandas as pd,numpy as np,sys
from pathlib import Path
root=Path(sys.argv[1]); folder=root/'data/processed/experiments/salh_group_kfold_v1'
master=pd.read_csv(root/'data/processed/modeling/master.csv').set_index('raw_row_id')
pred=pd.read_csv(folder/'inner_selected_predictions.csv')
items=pd.read_csv(folder/'model_artifacts.csv')
for row in items.itertuples(index=False):
 path=root/row.path; meta=json.loads(path.with_suffix('.json').read_text())
 with path.open('rb') as f: model=cloudpickle.load(f)
 assert model.memory is None
 if row.role=='outer_train_only':
  pp=pred.loc[(pred.target==row.target)&(pred.outer_fold==row.outer_fold)]
  np.testing.assert_allclose(model.predict(master.loc[pp.raw_row_id,meta['features']]),pp.y_pred,rtol=1e-10,atol=1e-8)
  assert set(meta['train_groups']).isdisjoint(meta['valid_groups'])
 else:
  assert meta['do_not_use_for_fold_evaluation'] and not meta['independent_test_evaluated']
  assert np.isfinite(model.predict(master.loc[meta['train_raw_row_ids'][:10],meta['features']])).all()
print('40 pipelines groupées rechargées ; toutes les prédictions des 20 modèles externes reproduites.')
"""
verification=subprocess.run([sys.executable,'-c',verifier,str(PROJECT_ROOT)],check=True,
    capture_output=True,text=True,env=dict(os.environ,OPENBLAS_NUM_THREADS='1',MPLCONFIGDIR='/tmp/salh-matplotlib'))
print(verification.stdout,flush=True)


,target,family,fresh_process_verified,max_prediction_difference
0,yield_strength,random_forest,True,0.0
1,uts,svr_rbf,True,0.0
2,elongation,random_forest,True,0.0
3,charpy_J,random_forest,True,0.0


40 pipelines groupées rechargées ; toutes les prédictions des 20 modèles externes reproduites.



In [8]:

access_rows=[]
for target in TARGETS:
    for outer in range(5):
        for inner in [None,0,1,2]:
            train,pool,valid=grouped_access(data,target,outer,inner)
            held=data.loc[(data.outer_fold==outer) | ((data.outer_fold!=outer)&
                (data[f'inner_fold_when_outer{outer}']==inner))] if inner is not None else data.loc[data.outer_fold==outer]
            assert set(train.group).isdisjoint(held.group) and set(pool.group).isdisjoint(held.group)
            access_rows.append({'target':target,'outer_fold':outer,'inner_fold':-1 if inner is None else inner,
                'labeled_train_n':len(train),'unlabeled_train_n':len(pool),'labeled_valid_n':len(valid),
                'train_groups':train.group.nunique(),'pool_groups':pool.group.nunique()})
show_saved(pd.DataFrame(access_rows),'grouped_semi_supervised_access_counts.csv',80)


,target,outer_fold,inner_fold,labeled_train_n,unlabeled_train_n,labeled_valid_n,train_groups,pool_groups
0,yield_strength,0,-1,574,651,132,34,18
1,yield_strength,0,0,333,509,241,22,14
2,yield_strength,0,1,427,391,147,23,12
3,yield_strength,0,2,388,402,186,23,10
4,yield_strength,1,-1,540,685,166,32,17
...,...,...,...,...,...,...,...,...
75,charpy_J,3,2,480,0,206,14,0
76,charpy_J,4,-1,670,0,191,19,0
77,charpy_J,4,0,457,0,213,13,0
78,charpy_J,4,1,438,0,232,12,0


## Traçabilité

Les deux répertoires modèles sont distincts : `salh_random_kfold_v1/` et
`salh_group_kfold_v1/`, chacun avec outer_selected, full_fit et selected_models.json.
Cette sauvegarde ne transforme pas un refit complet en modèle utilisable pour
valider des plis : réentraîner seulement sur leurs données autorisées.


In [9]:

source_nb=json.loads((PROJECT_ROOT/'notebooks/06_grouped_exports.ipynb').read_text())
source_text='\n\n'.join(''.join(c['source']) for c in source_nb['cells'])
manifest={'protocol_version':'salh-group-kfold-exports-v1','feature_set':PRIMARY_SET,
    'numeric_features':CORE16,'imputer':PRIMARY_IMPUTER,'cohort':'unchanged strict13','seed':SEED,
    'outer_inner':'existing exported 5 outer / 3 inner grouped folds from preparation',
    'outer_tuning':'saved grouped 16/median inner results, rechecked; baseline recomputed',
    'full_refit_cv':'GroupKFold5 labeled groups, shuffle=False','grids':PARAM_GRIDS,'versions':VERSIONS,
    'selection':'inner MAE; no choice using outer scores',
    'semisupervised':'inductive access prepared; no algorithm implemented',
    'raw_sha256':preparation_manifest['raw_sha256'],
    'preparation_manifest_sha256':hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest(),
    'diagnostic_manifest_sha256':hashlib.sha256((DIAGNOSTICS/'manifest.json').read_bytes()).hexdigest(),
    'notebook_source_sha256':hashlib.sha256(source_text.encode()).hexdigest(),
    'verified_reference_evaluations':int(scores.verified_previous_OOF.sum()),
    'verified_reference_rows':len(reference_predictions),
    'models_exported':len(artifact_rows),'elapsed_seconds':time.perf_counter()-STARTED,
    'files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('*.csv')},
    'model_files':{str(p.relative_to(PROJECT_ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
                   for p in MODELS.rglob('*') if p.is_file()}}
assert manifest['models_exported']==40
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+'\n')
display(pd.Series({k:manifest[k] for k in ['models_exported','verified_reference_evaluations','verified_reference_rows','elapsed_seconds']}))


models_exported                      40.000000
verified_reference_evaluations       80.000000
verified_reference_rows           11500.000000
elapsed_seconds                     115.892561
dtype: float64